In [10]:
import mlflow
import pandas as pd
import numpy as np
import re
import nltk
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


In [11]:
df=pd.read_csv('IMDB.csv')
df=df.sample(500)
df.to_csv('data.csv',index=False)
df.head()

,review,sentiment
93,"Just saw it....the story, the plot, the script...",negative
414,It's astonishing that some people saw this as ...,negative
140,This movie have 4 parts and every is around 17...,positive
303,Soul Calibur is more solid than it ever was......,positive
212,"Awwww....yes, it is heartwarming and all that ...",negative


In [12]:
def lemmatize(text):
    lemmatizer=WordNetLemmatizer()
    text=text.split()
    text=[lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)
def remove_stopwords(text):
    stop_words=set(stopwords.words('english'))
    text=[word for word in str(text).split() if word not in stop_words]
    return " ".join(text)
def remove_numbers(text):
    text=''.join([char for char in text if not char.isdigit()])
    return text
def lower_case(text):
    text=text.split()
    text=[word.lower() for word in text]
    return " ".join(text)
def remove_punctuation(text):
    text=re.sub('[%s]'% re.escape(string.punctuation),' ',text)
    text=text.replace('\n',' ')
    text=re.sub('\s+',' ',text.strip())
    return text
def removing_urls(text):
    url_patterns=re.compile(r'https?://\S+|www\.\S+')
    return url_patterns.sub(r'',text)

def normalize_text(df):
    try:
        df['review']=df['review'].apply(lower_case)
        df['review']=df['review'].apply(remove_stopwords)
        df['review']=df['review'].apply(remove_numbers)
        df['review']=df['review'].apply(remove_punctuation)
        df['review']=df['review'].apply(removing_urls)
        df['review']=df['review'].apply(lemmatize)
        return df
    except Exception as e:
        print(f"Error during text normalization: {e}")
        raise
    


In [14]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\maloj\AppData\Roaming\nltk_data...


True

In [15]:
df=normalize_text(df)
df.head()

,review,sentiment
93,saw story plot script make absolute sense samv...,negative
414,astonishing people saw art saw poorly filmed s...,negative
140,movie part every around minute long based true...,positive
303,soul calibur solid ever new character creation...,positive
212,awwww yes heartwarming unlucky family get adop...,negative


In [16]:
x=df['sentiment'].isin(['positive','negative'])
df=df[x]


In [17]:
df['sentiment']=df['sentiment'].map({'positive':1,'negative':0})
df.head()

,review,sentiment
93,saw story plot script make absolute sense samv...,0
414,astonishing people saw art saw poorly filmed s...,0
140,movie part every around minute long based true...,1
303,soul calibur solid ever new character creation...,1
212,awwww yes heartwarming unlucky family get adop...,0


In [29]:
vectorizer=TfidfVectorizer(max_features=200)
X=vectorizer.fit_transform(df['review'])
y=df['sentiment']




In [30]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

In [ ]:
import dagshub
import dotenv
import os
dotenv.load_dotenv()
mlflow.set_tracking_uri(os.getenv("mlflow_tracking_uri"))
dagshub.init(repo_owner=os.getenv("dagshub_repo_owner"), repo_name=os.getenv("dagshub_repo_name"), mlflow=True)

mlflow.set_experiment('Logistic Regression Experiment')


2026-02-21 12:20:35,622 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/malojighorpade/Capstone-Project "HTTP/1.1 200 OK"


Initialized MLflow to track repo "malojighorpade/Capstone-Project"

2026-02-21 12:20:35,628 - INFO - Initialized MLflow to track repo "malojighorpade/Capstone-Project"


Repository malojighorpade/Capstone-Project initialized!

2026-02-21 12:20:35,635 - INFO - Repository malojighorpade/Capstone-Project initialized!


<Experiment: artifact_location='mlflow-artifacts:/81f7a0a23a9c4ba7b4db82d2563e47dd', creation_time=1771655765159, experiment_id='0', last_update_time=1771655765159, lifecycle_stage='active', name='Logistic Regression Experiment', tags={}, workspace='default'>

In [32]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logging.info("Starting mlflow...")

with mlflow.start_run():
    start_time=time.time()
    try:
        logging.info("Logging processing parameters")
        mlflow.log_param('vectorizer','Bag of Words')
        mlflow.log_param('max_features',200)
        mlflow.log_param('test_size',0.3)
        logging.info("Training Logistic Regression model")

        model=LogisticRegression(max_iter=1000)
        model.fit(X_train,y_train)
        logging.info("Model training completed")
        model.fit(X_train,y_train)
        logging.info("Model training completed")

        logging.info("Evaluating model performance")
        y_pred=model.predict(X_test)
        accuracy=accuracy_score(y_test,y_pred)
        precision=precision_score(y_test,y_pred)
        recall=recall_score(y_test,y_pred)
        f1=f1_score(y_test,y_pred)

        logging.info("calculating metrics")
        mlflow.log_metric('accuracy',accuracy)
        mlflow.log_metric('precision',precision)
        mlflow.log_metric('recall',recall)
        mlflow.log_metric('f1_score',f1)
        end_time=time.time()
        mlflow.log_metric('training_time',(end_time-start_time))

        end_time=time.time()
        logging.info(f"Total training time: {end_time-start_time} seconds")
         
        logging.info(f"Accuracy:{accuracy}")
        logging.info(f"Precision:{precision}")
        logging.info(f"Recall:{recall}")
        logging.info(f"F1 Score:{f1}")

    except Exception as e:
        logging.error(f"Error during mlflow run: {e}")
        raise

2026-02-21 12:21:09,977 - INFO - Starting mlflow...
2026-02-21 12:21:10,623 - INFO - Logging processing parameters
2026-02-21 12:21:11,691 - INFO - Training Logistic Regression model
2026-02-21 12:21:11,721 - INFO - Model training completed
2026-02-21 12:21:11,727 - INFO - Model training completed
2026-02-21 12:21:11,728 - INFO - Evaluating model performance
2026-02-21 12:21:11,740 - INFO - calculating metrics
2026-02-21 12:21:13,491 - INFO - Total training time: 2.867150068283081 seconds
2026-02-21 12:21:13,492 - INFO - Accuracy:0.6933333333333334
2026-02-21 12:21:13,492 - INFO - Precision:0.6707317073170732
2026-02-21 12:21:13,493 - INFO - Recall:0.7432432432432432
2026-02-21 12:21:13,493 - INFO - F1 Score:0.7051282051282052


🏃 View run painted-stoat-125 at: https://dagshub.com/malojighorpade/Capstone-Project.mlflow/#/experiments/0/runs/6ef65251d12b472ab4dbfca6fbb01cfb
🧪 View experiment at: https://dagshub.com/malojighorpade/Capstone-Project.mlflow/#/experiments/0
